# Notebook 3: Evacuation Regression Analysis

This notebook runs the main regression models examining social and behavioral determinants of evacuation distance and destination choice.

**Paper reference:** Figures 1b, 1e, 3a, 3b — Logistic and linear regression results; Supplementary Tables S6–S20.

**Inputs:** `evacuees_for_regression_final.csv`, `colorado_SCI_ZIP_CBG.csv`

**Outputs:** Regression tables, coefficient plots


In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
import seaborn as sns

In [ ]:
from matplotlib import cm
from matplotlib.colors import Normalize

In [ ]:
SCI = pd.read_csv("colorado_SCI_ZIP_CBG.csv")

In [ ]:
SCI.head()

In [ ]:
# Check for duplicates in SCI based on 'cbg_user' and 'cbg_fr'
duplicates = SCI.duplicated(subset=['cbg_user', 'cbg_fr'], keep=False)
if duplicates.sum() > 0:
    print(f"There are {duplicates.sum()} duplicate rows in SCI based on 'cbg_user' and 'cbg_fr'.")

SCI_unique = SCI.groupby(['cbg_user', 'cbg_fr'], as_index=False)['scaled_sci'].mean()

In [ ]:
# Aggregate the duplicates by taking the mean of 'scaled_sci' for each unique pair of 'cbg_user' and 'cbg_fr'
SCI_unique = SCI.groupby(['cbg_user', 'cbg_fr'], as_index=False)['scaled_sci'].mean()

# Check the shape of the resulting dataframe to ensure duplicates are removed
print(f"Shape of SCI after removing duplicates: {SCI_unique.shape}")

In [ ]:
evacuees = pd.read_csv("evacuees_for_regression_final.csv")

In [ ]:
evacuees_newSCI = pd.merge(
    evacuees, 
    SCI_unique, 
    left_on=['pre_crisis_home_cbg', 'crisis_home_cbg'],
    right_on=['cbg_user', 'cbg_fr'], 
    how='left' 
    )

In [ ]:
evacuees_newSCI.to_csv("evacuees_newSCI_simulation.csv")

In [ ]:
# Drop rows where 'scaled_sci' is NaN
evacuees_newSCI_cleaned = evacuees_newSCI.dropna(subset=['scaled_sci'])

print(f"Number of rows after dropping missing 'scaled_sci': {evacuees_newSCI_cleaned.shape[0]}")

In [ ]:
filtered_rows_1 = evacuees_newSCI_cleaned[(evacuees_newSCI_cleaned['dist_epicentre'] <= 20000)]
filtered_rows_2 = evacuees_newSCI_cleaned[(evacuees_newSCI_cleaned['dist_epicentre'] >= 20000) & (evacuees_newSCI_cleaned['dist_epicentre'] <= 60000)]
filtered_rows_3 = evacuees_newSCI_cleaned[(evacuees_newSCI_cleaned['dist_epicentre'] >= 60000)]
count_1 = len(filtered_rows_1)
count_2 = len(filtered_rows_2)
count_3 = len(filtered_rows_3)
percent = ((count_2)/len(evacuees_newSCI_cleaned))*100
print(f"Number of rows where 'dist_epicentre' is between 0 and 20: {count_1}")
print(f"Number of rows where 'dist_epicentre' is between 20 and 60: {count_2}")
print(f"Number of rows where 'dist_epicentre' above: {count_3}")
percent

In [ ]:
filtered_rows_1 = evacuees_newSCI_cleaned[(evacuees_newSCI_cleaned['distance_OD'] <= 20000)]
filtered_rows_2 = evacuees_newSCI_cleaned[(evacuees_newSCI_cleaned['distance_OD'] >= 20000) & (evacuees_newSCI_cleaned['distance_OD'] <= 60000)]
filtered_rows_3 = evacuees_newSCI_cleaned[(evacuees_newSCI_cleaned['distance_OD'] >= 60000)]
count_1 = len(filtered_rows_1)
count_2 = len(filtered_rows_2)
count_3 = len(filtered_rows_3)
percent = ((count_2)/len(evacuees_newSCI_cleaned))*100
print(f"Number of rows where 'dist_epicentre' is between 0 and 20: {count_1}")
print(f"Number of rows where 'dist_epicentre' is between 20 and 60: {count_2}")
print(f"Number of rows where 'dist_epicentre' above: {count_3}")
percent

In [ ]:
plt.hist(filtered_data['distance_OD'], bins=20, color='grey', edgecolor='darkgrey')

# Add labels and title
plt.xlabel('Distance from Epicentre (meters)')
plt.ylabel('Percentage')
plt.title('Weighted Histogram of Destination Rate vs Distance from Epicentre (Evacuated Only)')

# Customize the plot
plt.grid(axis='y', linestyle='--', alpha=0.7)
sns.despine()

# Show the plot
plt.tight_layout()
plt.show()

In [ ]:
evacuees_newSCI_cleaned['log_dist_epicentre'] = evacuees_newSCI_cleaned['dist_epicentre'].apply(lambda x: np.log10(1 + x))
evacuees_newSCI_cleaned['log_distance_OD'] = evacuees_newSCI_cleaned['distance_OD'].apply(lambda x: np.log10(1 + x))
evacuees_newSCI_cleaned['log_D1_population_density'] = evacuees_newSCI_cleaned['D1_population_density'].apply(lambda x: np.log10(1 + x))
evacuees_newSCI_cleaned['log_Or_population_density'] = evacuees_newSCI_cleaned['Or_population_density'].apply(lambda x: np.log10(1 + x))
evacuees_newSCI_cleaned['log_Or_unemployment_rate'] = evacuees_newSCI_cleaned['Or_unemployment_rate'].apply(lambda x: np.log10(1 + x))
evacuees_newSCI_cleaned['log_Or_total_housing_units'] = evacuees_newSCI_cleaned['Or_total_housing_units'].apply(lambda x: np.log10(1 + x))
evacuees_newSCI_cleaned['log_radius_of_gyration'] = evacuees_newSCI_cleaned['radius_of_gyration'].apply(lambda x: np.log10(1 + x))
evacuees_newSCI_cleaned['log_Or_white_population'] = evacuees_newSCI_cleaned['Or_white_population'].apply(lambda x: np.log10(1 + x))
evacuees_newSCI_cleaned['log_black_population'] = evacuees_newSCI_cleaned['Or_black_population'].apply(lambda x: np.log10(1 + x))
evacuees_newSCI_cleaned['log_Or_asian_population'] = evacuees_newSCI_cleaned['Or_asian_population'].apply(lambda x: np.log10(1 + x))
evacuees_newSCI_cleaned['log_Or_all_others'] = evacuees_newSCI_cleaned['Or_all_others'].apply(lambda x: np.log10(1 + x))
evacuees_newSCI_cleaned['log_scaled_sci'] = evacuees_newSCI_cleaned['scaled_sci'].apply(np.log1p)

In [ ]:
df = evacuees_newSCI_cleaned.dropna()
from statsmodels.stats.outliers_influence import variance_inflation_factor
independent_vars = ['Or_population_density', 
                    'log_Or_unemployment_rate',
                    #'Or_total_housing_units_r', 
                    'radius_of_gyration',
                    #'log_Or_white_population',
                    #'log_black_population',
                    'log_Or_asian_population', 
                    'avg_entropy', 
                    'Or_median_household_income',
                    'Or_total_population', 
                    #'Or_occupied_housing_units',
                    'Or_education_atleast_one_degree', 
                    'dist_epicentre',
                    'log_Or_all_others'
]

# Calculate VIF for the independent variables
X = df[independent_vars]
vif_data = pd.DataFrame()
vif_data["feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print(vif_data)

# Final Linear regressions

## Distributions

In [ ]:
df['Or_unemployment_rate'].hist()

In [ ]:
df['Or_asian_population'].hist()

In [ ]:
df['Or_black_population'].hist()

In [ ]:
df['Or_white_population'].hist()

In [ ]:
df['distance_OD'].hist()

In [ ]:
df['Or_median_household_income'].hist()

In [ ]:
df['avg_entropy'].hist()

In [ ]:
df['radius_of_gyration'].hist()

In [ ]:
df['dist_epicentre'].hist()

In [ ]:
df['Or_unemployment'].hist()

In [ ]:
df['Or_occupied_housing_units'].hist()

# Regressions

In [ ]:
from sklearn.preprocessing import StandardScaler

## Distance

In [ ]:
df_dist = evacuees_newSCI_cleaned.dropna()
Y = df_dist['log_distance_OD'] #already log transformed

independent_vars = ['Or_population_density', 
                    'Or_unemployment_rate',
                    'radius_of_gyration',
                    'log_Or_white_population',
                    'log_black_population',
                    'log_Or_asian_population', 
                    'avg_entropy', 
                    'Or_median_household_income',
                    'Or_education_atleast_one_degree', 
                    'dist_epicentre'
]

# Standardize the features
scaler = StandardScaler()
df_dist[independent_vars] = scaler.fit_transform(df_dist[independent_vars])

# Prepare the independent variables DataFrame
X = df_dist[independent_vars]

# Add a constant term (intercept) to the independent variables
X_with_const = sm.add_constant(X)

# Perform the regression
model_distance_OD = sm.OLS(Y, X_with_const).fit()

# Print the regression summary
print("Regression summary for Distance between Origin and Destination:\n")
print(model_distance_OD.summary())


In [ ]:
# Define the different models (D1, D3, Final)
models_dist = {
    'Demographics (D1)': ['Or_population_density', 
                          #'log_Or_unemployment_rate',
                          #'log_Or_white_population',
                          'log_black_population',
                          'log_Or_asian_population', 
                          'Or_median_household_income',
                          'Or_education_atleast_one_degree'
                         ],
    
    'D1 + Distance from fire (D3)': ['Or_population_density', 
                                     #'log_Or_unemployment_rate',
                                     #'log_Or_white_population',
                                     'log_black_population',
                                     'log_Or_asian_population', 
                                     'Or_median_household_income',
                                     'Or_education_atleast_one_degree', 
                                     'dist_epicentre'],
    
    'D1 + D3 + Pre-disaster mobility': ['Or_population_density', 
                                        #'log_Or_unemployment_rate',
                                        #'log_Or_white_population',
                                        'log_black_population',
                                        'log_Or_asian_population', 
                                        'Or_median_household_income',
                                        'Or_education_atleast_one_degree', 
                                        'dist_epicentre',
                                        'avg_entropy',
                                        'radius_of_gyration']
}

# Initialize lists to store coefficients, errors, and R-squared values for plotting
coef_data_distance_OD = []
r_squared_values = []

# Loop through each model setup
for model_name, independent_vars in models_dist.items():
    # Standardize the features for the current model within the loop
    X = df_dist[independent_vars].copy()
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    # Add a constant term (intercept)
    X_with_const = sm.add_constant(X)

    # Perform the regression
    model_distance_OD = sm.OLS(df_dist['log_distance_OD'], X_with_const).fit()
    
    # Collect coefficients, errors, and model name for plotting
    coef_df = pd.DataFrame({
        'Model': model_name,
        'Variable': independent_vars,  # Use actual variable names
        'Coefficient': model_distance_OD.params[1:],  # Exclude the intercept
        'Standard_Error': model_distance_OD.bse[1:],  # Exclude the intercept
        'PValue': model_distance_OD.pvalues[1:]  # Exclude the intercept
    })
    coef_data_distance_OD.append(coef_df)

    # Store R-squared value for the bar graph
    r_squared_values.append({
        'Model': model_name,
        'R-squared': model_distance_OD.rsquared
    })

# Combine all model results into a single DataFrame
combined_coef_df_distance_OD = pd.concat(coef_data_distance_OD)

# Convert R-squared values to a DataFrame
r_squared_df = pd.DataFrame(r_squared_values)

# # Define colors and markers for models
# colors = sns.color_palette("Set2", len(models_dist))
# markers = ['o', 's', '^']

# # Function to plot coefficients with error bars for multiple models
# def plot_coefficients_with_error_bars(combined_coef_df, dependent_variable_name):
#     fig, ax = plt.subplots(figsize=(10, 8))
    
#     for i, model_name in enumerate(combined_coef_df['Model'].unique()):
#         data = combined_coef_df[combined_coef_df['Model'] == model_name]
#         ax.errorbar(data['Coefficient'], data['Variable'], xerr=data['Standard_Error'], fmt=markers[i], color=colors[i],
#                     ecolor='darkgrey', elinewidth=1, capsize=5, markersize=10, linestyle='None',
#                     markerfacecolor='none', markeredgewidth=1, label=model_name)

#     ax.axvline(0, color='black', linewidth=0.5, linestyle='--')
#     ax.set_xlabel('Beta Coefficients')
#     ax.set_ylabel('Variables')
#     ax.set_title(f'Beta Coefficients for {dependent_variable_name}')
#     plt.legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left')
#     plt.tight_layout()
#     plt.show()


# Print regression summary for the last model as an example (optional)
print("Regression summary for the final model:\n")
print(model_distance_OD.summary())

In [ ]:
# Plot the R-squared horizontal bar graph
plt.figure(figsize=(6, 3))
sns.barplot(y='Model', x='R-squared', data=r_squared_df, palette='Set2', orient='h')
plt.title("R-squared values for Different Models")
plt.xlabel("R-squared")
plt.ylabel("Model")
plt.tight_layout()

# Force rendering of the plot
plt.draw()

# Save the R-squared plot in 300 DPI
plt.savefig('r_squared_models.png', dpi=300)

plt.show()

In [ ]:
# Function to plot coefficients with error bars for multiple models
def plot_coefficients_with_error_bars(combined_coef_df, dependent_variable_name):
    fig, ax = plt.subplots(figsize=(10, 8))
    
    for i, model_name in enumerate(combined_coef_df['Model'].unique()):
        data = combined_coef_df[combined_coef_df['Model'] == model_name]
        ax.errorbar(data['Coefficient'], data['Variable'], xerr=data['Standard_Error'], fmt=markers[i], color=colors[i],
                    ecolor='darkgrey', elinewidth=1, capsize=5, markersize=10, linestyle='None',
                    markerfacecolor='none', markeredgewidth=1, label=model_name)

    ax.axvline(0, color='black', linewidth=0.5, linestyle='--')
    ax.set_xlabel('Beta Coefficients')
    ax.set_ylabel('Variables')
    ax.set_title(f'Beta Coefficients for {dependent_variable_name}')
    plt.legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()

    # Save the figure in 300 DPI
    plt.savefig(f'coefficients_{dependent_variable_name}.png', dpi=300)

    plt.show()

# SCI and Homophily

### Plain regression

In [ ]:
df_homo = evacuees_newSCI_cleaned.dropna()
Y = df_homo['homophily'] 

independent_vars = [
                        # 'Or_population_density', 
                      #'log_Or_unemployment_rate', 
                     #'log_Or_white_population',
                       #'log_black_population',
#                      #'log_Or_asian_population',
#                      'log_Or_all_others',
                      'Or_median_household_income',
                      #'Or_education_atleast_one_degree', 
                      #'dist_epicentre',
                      'log_distance_OD',
                    #  'avg_entropy',
                    # 'radius_of_gyration'
                    ]
# Standardize the features
scaler = StandardScaler()
df_homo[independent_vars] = scaler.fit_transform(df_homo[independent_vars])

# Prepare the independent variables DataFrame
X = df_homo[independent_vars]

# Add a constant term (intercept) to the independent variables
X_with_const = sm.add_constant(X)

# Perform the regression
model_homo= sm.OLS(Y, X_with_const).fit()

# Print the regression summary
print("Regression summary for Similarity:\n")
print(model_homo.summary())


In [ ]:
df_SCI = evacuees_newSCI_cleaned.dropna()
Y = df_SCI['log_scaled_sci'] 

independent_vars = [
    # 'Or_population_density', 
                     # 'log_Or_unemployment_rate', 
                  #'log_Or_white_population',
                    'log_black_population',
    #                  'log_Or_all_others',
    #                  #'log_Or_asian_population',  
                      #'Or_median_household_income',
                    # 'Or_education_atleast_one_degree', 
                      #'dist_epicentre',
                      'log_distance_OD',
    #                  'avg_entropy',
    #                 'radius_of_gyration'
                    ]
# Standardize the features
scaler = StandardScaler()
df_SCI[independent_vars] = scaler.fit_transform(df_SCI[independent_vars])

# Prepare the independent variables DataFrame
X = df_SCI[independent_vars]

# Add a constant term (intercept) to the independent variables
X_with_const = sm.add_constant(X)

# Perform the regression
model_SCI= sm.OLS(Y, X_with_const).fit()

# Print the regression summary
print("Regression summary for Familiarity:\n")
print(model_SCI.summary())


## multiple regression with plot

In [ ]:
# Clean the DataFrame by dropping any rows with NaN values
df = evacuees_newSCI_cleaned.dropna()
df['log_scaled_sci'] = df['scaled_sci'].apply(np.log1p)

# Define the models with dependent and independent variables
models = [
    {"y_var": "log_scaled_sci", "x_vars": ["log_distance_OD",  "log_Or_white_population"]},
    {"y_var": "log_scaled_sci", "x_vars": ["log_distance_OD",  "log_black_population"]},
    {"y_var": "log_scaled_sci", "x_vars": ["log_distance_OD",  "Or_median_household_income"]},
    {"y_var": "homophily", "x_vars": ["log_distance_OD",  "log_Or_white_population"]},
    {"y_var": "homophily", "x_vars": ["log_distance_OD",  "log_black_population"]},
    {"y_var": "homophily", "x_vars": ["log_distance_OD",  "Or_median_household_income"]},
]

# Initialize an empty list to store results
results_list = []

# Loop through each model configuration
for model_info in models:
    y_var = model_info["y_var"]
    x_vars = model_info["x_vars"]
    
    # Define Y and X using the cleaned DataFrame
    Y = df[[y_var]]  # Convert Y to a DataFrame for scaling
    X = df[x_vars]
    
    # Standardize X variables
    x_scaler = StandardScaler()
    X_scaled = x_scaler.fit_transform(X)
    X_scaled = sm.add_constant(X_scaled)  # Add constant (intercept) term
    
    # Fit the regression model
    model = sm.OLS(Y, X_scaled).fit()
    
    # Extract confidence intervals
    conf_int = model.conf_int(alpha=0.05)  # 95% confidence intervals

    # Extract model summary statistics
    r_squared = model.rsquared
    for i, (var_name, coef, std_err, t_value, p_value) in enumerate(
        zip(x_vars, model.params[1:], model.bse[1:], model.tvalues[1:], model.pvalues[1:])
    ):
        lower_ci, upper_ci = conf_int.iloc[i + 1]  # Adjust index to skip the intercept
        results_list.append({
            "y_variable": y_var,
            "x_variable": var_name,
            "R_squared": r_squared,
            "Coefficient": round(coef, 4),
            "Std_Error": round(std_err, 4),
            "t_value": round(t_value, 4),
            "p_value": round(p_value, 4),
            "CI_Lower": round(lower_ci, 4),  # Lower bound of confidence interval
            "CI_Upper": round(upper_ci, 4)   # Upper bound of confidence interval
        })


# Convert results to a DataFrame
results_df = pd.DataFrame(results_list)

# Display the results
results_df

In [ ]:
def plot_selected_coefficients(df, y_vars, selected_x_vars, significance_level=0.05):
    # Define colors for significant and insignificant variables
    color_mapping = {
        'log_scaled_sci': 'orange',  # Orange for significant coefficients in log_scaled_sci models
        'homophily': 'steelblue'    # Steel blue for significant coefficients in homophily models
    }
    insignificance_color = 'grey'

    for y_var in y_vars:
        # Filter for the current y variable and selected x variables
        y_df = df[(df['y_variable'] == y_var) & (df['x_variable'].isin(selected_x_vars))]

        # Debugging: Check the filtered data
        print(f"\nData for {y_var}:")
        print(y_df[['x_variable', 'Coefficient', 'CI_Lower', 'CI_Upper', 'p_value']])

        # Ensure the x_variables are ordered as specified
        y_df['x_variable'] = pd.Categorical(
            y_df['x_variable'], categories=selected_x_vars, ordered=True
        )
        y_df = y_df.sort_values('x_variable')

        # Plot
        fig, ax = plt.subplots(figsize=(5, 2))
        ax.invert_yaxis()

        # Plot each coefficient with error bars using precomputed confidence intervals
        for i, row in y_df.iterrows():
            # Choose color based on significance
            plot_color = color_mapping.get(y_var, 'green') if row['p_value'] < significance_level else insignificance_color
            
            # Calculate error bar values using confidence intervals
            ci_lower = row['Coefficient'] - row['CI_Lower']
            ci_upper = row['CI_Upper'] - row['Coefficient']

            # Debugging: Print CI values
            print(f"x_variable: {row['x_variable']}, Coefficient: {row['Coefficient']}, CI_Lower: {ci_lower}, CI_Upper: {ci_upper}")
            
            ax.errorbar(row['Coefficient'], row['x_variable'], 
                        xerr=[[ci_lower], [ci_upper]], fmt='v', color=plot_color,
                        ecolor='black', elinewidth=0.8, capsize=4, markersize=8, linestyle='None',
                        markerfacecolor=plot_color, markeredgewidth=1.5, alpha=0.8)

        # Customize plot appearance
        ax.axvline(0, color='grey', linewidth=0.8, linestyle='--')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_visible(False)
        ax.spines['bottom'].set_visible(False)
        ax.xaxis.set_ticks_position('bottom')
        ax.yaxis.set_ticks_position('left')

        # Adjust x-axis limits for consistency
        max_limit = max(abs(y_df['CI_Upper'].max()), abs(y_df['CI_Lower'].min()))
        ax.set_xlim(-max_limit, max_limit)

        plt.tight_layout()

        # Save the plot
        plot_filename = f"{y_var}_coeff_plot.png"
        plt.savefig(plot_filename, dpi=300, bbox_inches='tight')
        print(f"Plot saved as {plot_filename}")
        plt.show()

# Example Usage
y_variables = ['log_scaled_sci', 'homophily']  # Correct y_variable names
selected_x_variables = ['log_Or_white_population', 'log_black_population', 'Or_median_household_income']

# Assuming `results_df` is your DataFrame containing precomputed confidence intervals
plot_selected_coefficients(results_df, y_variables, selected_x_variables)

## Stargazer

In [ ]:
df_sci = evacuees_newSCI_cleaned.dropna()
df_sci['log_scaled_sci'] = df_sci['scaled_sci'].apply(np.log1p)

In [ ]:
Y = df_sci['log_scaled_sci'] 
independent_vars = ['Or_median_household_income',
                    'log_distance_OD',
                    ]
scaler = StandardScaler()
df_sci[independent_vars] = scaler.fit_transform(df_sci[independent_vars])
X = df_sci[independent_vars]
X_with_const = sm.add_constant(X)
model_sci_income= sm.OLS(Y, X_with_const).fit()


In [ ]:
params = model_sci_income.params.round(4)
pvalues = model_sci_income.pvalues.round(4)
conf_int = model_sci_income.conf_int().round(4)

variable_names = {
    'Or_median_household_income': 'Median Household Income',
    'log_scaled_sci': 'Connectedness',
    'log_distance_OD': 'Evacuation Distance'
}
latex_output = r"""
\begin{table}[!htbp] \centering
  \caption{Regression Results: Connectedness = Evacuation Distance + Median Income}
\begin{tabular}{@{\extracolsep{5pt}}lc}
\hline \hline
& \multicolumn{1}{c}{\textit{Dependent variable: Connectedness}} \\
\cline{2-2}
\\[-1.8ex] & \textbf{Model 1} \\
\hline \\[-1.8ex]
"""

for var, coef in params.items():
    # Get the renamed variable, or default to the original name
    var_name = variable_names.get(var, var).title()
    pval = pvalues[var]
    lower_ci, upper_ci = conf_int.loc[var]
    
    latex_output += f"{var_name} & {coef} ($p={pval}$) \\\\\n"
    latex_output += f"& [{lower_ci}, {upper_ci}] \\\\\n"

latex_output += r"""
\hline \\[-1.8ex]
Observations & %d \\
$R^2$ & %.3f \\
Adjusted $R^2$ & %.3f \\
\hline
\hline \\[-1.8ex]
\textit{Note:} & \multicolumn{1}{r}{$^{*}$p$<$0.1; $^{**}$p$<$0.05; $^{***}$p$<$0.01} \\
\end{tabular}
\end{table}
""" % (model_sci_income.nobs, model_sci_income.rsquared, model_sci_income.rsquared_adj)

print(latex_output)

In [ ]:
Y = df_sci['log_scaled_sci'] 
independent_vars = ['log_Or_white_population',
                    'log_distance_OD',
                    ]
scaler = StandardScaler()
df_sci[independent_vars] = scaler.fit_transform(df_sci[independent_vars])
X = df_sci[independent_vars]
X_with_const = sm.add_constant(X)
model_sci_white= sm.OLS(Y, X_with_const).fit()


In [ ]:
params = model_sci_white.params.round(4)
pvalues = model_sci_white.pvalues.round(4)
conf_int = model_sci_white.conf_int().round(4)

variable_names = {
    'log_Or_white_population': 'White percent',
    'log_scaled_sci': 'Connectedness',
    'log_distance_OD': 'Evacuation Distance'
}

latex_output = r"""
\begin{table}[!htbp] \centering
  \caption{Regression Results: Connectedness = Evacuation Distance + White Percent}
\begin{tabular}{@{\extracolsep{5pt}}lc}
\hline \hline
& \multicolumn{1}{c}{\textit{Dependent variable: Connectedness}} \\
\cline{2-2}
\\[-1.8ex] & \textbf{Model 1} \\
\hline \\[-1.8ex]
"""

for var, coef in params.items():
    # Get the renamed variable, or default to the original name
    var_name = variable_names.get(var, var).title()
    pval = pvalues[var]
    lower_ci, upper_ci = conf_int.loc[var]
    
    # Add the row to the table
    latex_output += f"{var_name} & {coef} ($p={pval}$) \\\\\n"
    latex_output += f"& [{lower_ci}, {upper_ci}] \\\\\n"

latex_output += r"""
\hline \\[-1.8ex]
Observations & %d \\
$R^2$ & %.3f \\
Adjusted $R^2$ & %.3f \\
\hline
\hline \\[-1.8ex]
\textit{Note:} & \multicolumn{1}{r}{$^{*}$p$<$0.1; $^{**}$p$<$0.05; $^{***}$p$<$0.01} \\
\end{tabular}
\end{table}
""" % (model_sci_white.nobs, model_sci_white.rsquared, model_sci_white.rsquared_adj)

print(latex_output)

In [ ]:
Y = df_sci['log_scaled_sci'] 
independent_vars = ['log_black_population',
                    'log_distance_OD',
                    ]
scaler = StandardScaler()
df_sci[independent_vars] = scaler.fit_transform(df_sci[independent_vars])
X = df_sci[independent_vars]
X_with_const = sm.add_constant(X)
model_sci_black= sm.OLS(Y, X_with_const).fit()


In [ ]:
params = model_sci_black.params.round(4)
pvalues = model_sci_black.pvalues.round(4)
conf_int = model_sci_black.conf_int().round(4)

variable_names = {
    'log_black_population': 'Black percent',
    'log_scaled_sci': 'Connectedness',
    'log_distance_OD': 'Evacuation Distance'
}

latex_output = r"""
\begin{table}[!htbp] \centering
  \caption{Regression Results: Connectedness = Evacuation Distance + Black Percent}
\begin{tabular}{@{\extracolsep{5pt}}lc}
\hline \hline
& \multicolumn{1}{c}{\textit{Dependent variable: Connectedness}} \\
\cline{2-2}
\\[-1.8ex] & \textbf{Model 1} \\
\hline \\[-1.8ex]
"""

for var, coef in params.items():
    # Get the renamed variable, or default to the original name
    var_name = variable_names.get(var, var).title()
    pval = pvalues[var]
    lower_ci, upper_ci = conf_int.loc[var]
    
    latex_output += f"{var_name} & {coef} ($p={pval}$) \\\\\n"
    latex_output += f"& [{lower_ci}, {upper_ci}] \\\\\n"

latex_output += r"""
\hline \\[-1.8ex]
Observations & %d \\
$R^2$ & %.3f \\
Adjusted $R^2$ & %.3f \\
\hline
\hline \\[-1.8ex]
\textit{Note:} & \multicolumn{1}{r}{$^{*}$p$<$0.1; $^{**}$p$<$0.05; $^{***}$p$<$0.01} \\
\end{tabular}
\end{table}
""" % (model_sci_black.nobs, model_sci_black.rsquared, model_sci_black.rsquared_adj)

print(latex_output)

In [ ]:
df_homo = evacuees_newSCI_cleaned.dropna()
Y = df_homo['homophily'] 
independent_vars = ['Or_median_household_income',
                    'log_distance_OD',
                    ]
scaler = StandardScaler()
df_homo[independent_vars] = scaler.fit_transform(df_homo[independent_vars])
X = df_homo[independent_vars]
X_with_const = sm.add_constant(X)
model_homo_income= sm.OLS(Y, X_with_const).fit()


In [ ]:
params = model_homo_income.params.round(4)
pvalues = model_homo_income.pvalues.round(4)
conf_int = model_homo_income.conf_int().round(4)

variable_names = {
    'Or_median_household_income': 'Median Income',
    'homophily': 'Similarity',
    'log_distance_OD': 'Evacuation Distance'
}

latex_output = r"""
\begin{table}[!htbp] \centering
  \caption{Regression Results: Similarity = Evacuation Distance + Median Income}
\begin{tabular}{@{\extracolsep{5pt}}lc}
\hline \hline
& \multicolumn{1}{c}{\textit{Dependent variable: Similarity}} \\
\cline{2-2}
\\[-1.8ex] & \textbf{Model 1} \\
\hline \\[-1.8ex]
"""

for var, coef in params.items():
    # Get the renamed variable, or default to the original name
    var_name = variable_names.get(var, var).title()
    pval = pvalues[var]
    lower_ci, upper_ci = conf_int.loc[var]
    
    latex_output += f"{var_name} & {coef} ($p={pval}$) \\\\\n"
    latex_output += f"& [{lower_ci}, {upper_ci}] \\\\\n"

latex_output += r"""
\hline \\[-1.8ex]
Observations & %d \\
$R^2$ & %.3f \\
Adjusted $R^2$ & %.3f \\
\hline
\hline \\[-1.8ex]
\textit{Note:} & \multicolumn{1}{r}{$^{*}$p$<$0.1; $^{**}$p$<$0.05; $^{***}$p$<$0.01} \\
\end{tabular}
\end{table}
""" % (model_homo_income.nobs, model_homo_income.rsquared, model_homo_income.rsquared_adj)

print(latex_output)

In [ ]:
Y = df_homo['homophily'] 
independent_vars = ['log_Or_white_population',
                    'log_distance_OD',
                    ]
scaler = StandardScaler()
df_homo[independent_vars] = scaler.fit_transform(df_homo[independent_vars])
X = df_homo[independent_vars]
X_with_const = sm.add_constant(X)
model_homo_white= sm.OLS(Y, X_with_const).fit()


In [ ]:
params = model_homo_white.params.round(4)
pvalues = model_homo_white.pvalues.round(4)
conf_int = model_homo_white.conf_int().round(4)

variable_names = {
    'log_Or_white_population': 'White percent',
    'homophily': 'Similarity',
    'log_distance_OD': 'Evacuation Distance'
}

latex_output = r"""
\begin{table}[!htbp] \centering
  \caption{Regression Results: Similarity = Evacuation Distance + White percent}
\begin{tabular}{@{\extracolsep{5pt}}lc}
\hline \hline
& \multicolumn{1}{c}{\textit{Dependent variable: Similarity}} \\
\cline{2-2}
\\[-1.8ex] & \textbf{Model 1} \\
\hline \\[-1.8ex]
"""

for var, coef in params.items():
    # Get the renamed variable, or default to the original name
    var_name = variable_names.get(var, var).title()
    pval = pvalues[var]
    lower_ci, upper_ci = conf_int.loc[var]
    
    latex_output += f"{var_name} & {coef} ($p={pval}$) \\\\\n"
    latex_output += f"& [{lower_ci}, {upper_ci}] \\\\\n"

latex_output += r"""
\hline \\[-1.8ex]
Observations & %d \\
$R^2$ & %.3f \\
Adjusted $R^2$ & %.3f \\
\hline
\hline \\[-1.8ex]
\textit{Note:} & \multicolumn{1}{r}{$^{*}$p$<$0.1; $^{**}$p$<$0.05; $^{***}$p$<$0.01} \\
\end{tabular}
\end{table}
""" % (model_homo_white.nobs, model_homo_white.rsquared, model_homo_white.rsquared_adj)

print(latex_output)

In [ ]:
Y = df_homo['homophily'] 
independent_vars = ['log_black_population',
                    'log_distance_OD',
                    ]
scaler = StandardScaler()
df_homo[independent_vars] = scaler.fit_transform(df_homo[independent_vars])
X = df_homo[independent_vars]
X_with_const = sm.add_constant(X)
model_homo_black= sm.OLS(Y, X_with_const).fit()


In [ ]:
params = model_homo_black.params.round(4)
pvalues = model_homo_black.pvalues.round(4)
conf_int = model_homo_black.conf_int().round(4)

variable_names = {
    'log_black_population': 'Black percent',
    'homophily': 'Similarity',
    'log_distance_OD': 'Evacuation Distance'
}

latex_output = r"""
\begin{table}[!htbp] \centering
  \caption{Regression Results: Similarity = Evacuation Distance + Black percent}
\begin{tabular}{@{\extracolsep{5pt}}lc}
\hline \hline
& \multicolumn{1}{c}{\textit{Dependent variable: Similarity}} \\
\cline{2-2}
\\[-1.8ex] & \textbf{Model 1} \\
\hline \\[-1.8ex]
"""

for var, coef in params.items():
    # Get the renamed variable, or default to the original name
    var_name = variable_names.get(var, var).title()
    pval = pvalues[var]
    lower_ci, upper_ci = conf_int.loc[var]
    
    latex_output += f"{var_name} & {coef} ($p={pval}$) \\\\\n"
    latex_output += f"& [{lower_ci}, {upper_ci}] \\\\\n"

latex_output += r"""
\hline \\[-1.8ex]
Observations & %d \\
$R^2$ & %.3f \\
Adjusted $R^2$ & %.3f \\
\hline
\hline \\[-1.8ex]
\textit{Note:} & \multicolumn{1}{r}{$^{*}$p$<$0.1; $^{**}$p$<$0.05; $^{***}$p$<$0.01} \\
\end{tabular}
\end{table}
""" % (model_homo_black.nobs, model_homo_black.rsquared, model_homo_black.rsquared_adj)

print(latex_output)

In [ ]:
df_dist = evacuees_newSCI_cleaned.dropna()
Y = df_dist['log_distance_OD'] 
independent_vars = ['Or_population_density', 
                    'Or_unemployment_rate',
                    'radius_of_gyration',
                    'log_Or_white_population',
                    'log_black_population',
                    'log_Or_asian_population', 
                    'avg_entropy', 
                    'Or_median_household_income',
                    'Or_education_atleast_one_degree', 
                    'dist_epicentre'
]

scaler = StandardScaler()
df_dist[independent_vars] = scaler.fit_transform(df_dist[independent_vars])

X = df_dist[independent_vars]

X_with_const = sm.add_constant(X)

model_distance_OD_demo_dist_pdm = sm.OLS(Y, X_with_const).fit()


In [ ]:
Y = df_dist['log_distance_OD'] #already log transformed

independent_vars = ['Or_population_density', 
                    'Or_unemployment_rate',
                    'log_Or_white_population',
                    'log_black_population',
                    'log_Or_asian_population',  
                    'Or_median_household_income',
                    'Or_education_atleast_one_degree', 
                    'dist_epicentre'
]

scaler = StandardScaler()
df_dist[independent_vars] = scaler.fit_transform(df_dist[independent_vars])

X = df_dist[independent_vars]

X_with_const = sm.add_constant(X)

model_distance_OD_demo_dist = sm.OLS(Y, X_with_const).fit()

In [ ]:
Y = df_dist['log_distance_OD'] #already log transformed

independent_vars = ['Or_population_density', 
                    'Or_unemployment_rate',
                    'log_Or_white_population',
                    'log_black_population',
                    'log_Or_asian_population',  
                    'Or_median_household_income',
                    'Or_education_atleast_one_degree', 
]

scaler = StandardScaler()
df_dist[independent_vars] = scaler.fit_transform(df_dist[independent_vars])

X = df_dist[independent_vars]

X_with_const = sm.add_constant(X)

model_distance_OD_demo = sm.OLS(Y, X_with_const).fit()

In [ ]:
from stargazer.stargazer import Stargazer, LineLocation

In [ ]:
# Extract model results
params = model_distance_OD_demo.params.round(4)
pvalues = model_distance_OD_demo.pvalues.round(4)
conf_int = model_distance_OD_demo.conf_int().round(4)

# Define variable renaming (same as in Stargazer)
variable_names = {
    'Or_population_density': 'Population Density',
    'Or_unemployment_rate': 'Unemployment',
    'log_Or_white_population': 'White Percent',
    'log_black_population': 'Black Percent',
    'log_Or_asian_population': 'Asian Percent',
    'Or_median_household_income': 'Median Household Income',
    'Or_education_atleast_one_degree': 'Education Percent',
}

# Start the LaTeX table
latex_output = r"""
\begin{table}[!htbp] \centering
  \caption{Regression Results: Evacuation Distance ~ Sociodemographics}
\begin{tabular}{@{\extracolsep{5pt}}lc}
\hline \hline
& \multicolumn{1}{c}{\textit{Dependent variable: Evacuation Distance}} \\
\cline{2-2}
\\[-1.8ex] & \textbf{Model 1} \\
\hline \\[-1.8ex]
"""

# Add coefficients, p-values, and confidence intervals
for var, coef in params.items():
    # Get the renamed variable, or default to the original name
    var_name = variable_names.get(var, var).title()
    pval = pvalues[var]
    lower_ci, upper_ci = conf_int.loc[var]
    
    # Add the row to the table
    latex_output += f"{var_name} & {coef} ($p={pval}$) \\\\\n"
    latex_output += f"& [{lower_ci}, {upper_ci}] \\\\\n"

# Add table footer
latex_output += r"""
\hline \\[-1.8ex]
Observations & %d \\
$R^2$ & %.3f \\
Adjusted $R^2$ & %.3f \\
\hline
\hline \\[-1.8ex]
\textit{Note:} & \multicolumn{1}{r}{$^{*}$p$<$0.1; $^{**}$p$<$0.05; $^{***}$p$<$0.01} \\
\end{tabular}
\end{table}
""" % (model_distance_OD_demo.nobs, model_distance_OD_demo.rsquared, model_distance_OD_demo.rsquared_adj)

# Print the LaTeX output
print(latex_output)

In [ ]:
# Extract model results
params = model_distance_OD_demo_dist.params.round(4)
pvalues = model_distance_OD_demo_dist.pvalues.round(4)
conf_int = model_distance_OD_demo_dist.conf_int().round(4)

# Define variable renaming (same as in Stargazer)
variable_names = {
    'Or_population_density': 'Population Density',
    'Or_unemployment_rate': 'Unemployment',
    'log_Or_white_population': 'White Percent',
    'log_black_population': 'Black Percent',
    'log_Or_asian_population': 'Asian Percent',
    'Or_median_household_income': 'Median Household Income',
    'Or_education_atleast_one_degree': 'Education Percent',
    'dist_epicentre': 'Distance from Fire'
}

# Start the LaTeX table
latex_output = r"""
\begin{table}[!htbp] \centering
  \caption{Regression Results: Evacuation Distance ~ Sociodemographics}
\begin{tabular}{@{\extracolsep{5pt}}lc}
\hline \hline
& \multicolumn{1}{c}{\textit{Dependent variable: Evacuation Distance}} \\
\cline{2-2}
\\[-1.8ex] & \textbf{Model 1} \\
\hline \\[-1.8ex]
"""

# Add coefficients, p-values, and confidence intervals
for var, coef in params.items():
    # Get the renamed variable, or default to the original name
    var_name = variable_names.get(var, var).title()
    pval = pvalues[var]
    lower_ci, upper_ci = conf_int.loc[var]
    
    # Add the row to the table
    latex_output += f"{var_name} & {coef} ($p={pval}$) \\\\\n"
    latex_output += f"& [{lower_ci}, {upper_ci}] \\\\\n"

# Add table footer
latex_output += r"""
\hline \\[-1.8ex]
Observations & %d \\
$R^2$ & %.3f \\
Adjusted $R^2$ & %.3f \\
\hline
\hline \\[-1.8ex]
\textit{Note:} & \multicolumn{1}{r}{$^{*}$p$<$0.1; $^{**}$p$<$0.05; $^{***}$p$<$0.01} \\
\end{tabular}
\end{table}
""" % (model_distance_OD_demo_dist.nobs, model_distance_OD_demo_dist.rsquared, model_distance_OD_demo_dist.rsquared_adj)

# Print the LaTeX output
print(latex_output)

In [ ]:
stargazer = Stargazer([model_distance_OD_demo_dist_pdm])
pvalues = model_distance_OD_demo_dist_pdm.pvalues.round(4)
stargazer.covariate_order(['Or_population_density', 
                    'Or_unemployment_rate',
                    'radius_of_gyration',
                    'log_Or_white_population',
                    'log_black_population',
                    'log_Or_asian_population', 
                    'avg_entropy', 
                    'Or_median_household_income',
                    'Or_education_atleast_one_degree', 
                    'dist_epicentre'])
stargazer.rename_covariates({'Or_population_density': "Population Density", 
                    'Or_unemployment_rate': 'Unemployment',
                    'radius_of_gyration': "Radius of gyration",
                    'log_Or_white_population': "White Percent",
                    'log_black_population':"Black Percent",
                    'log_Or_asian_population': "Asian Percent" , 
                    'avg_entropy': "Entropy", 
                    'Or_median_household_income': "Median household income",
                    'Or_education_atleast_one_degree': "Education Percent", 
                    'dist_epicentre': "Distance from Fire"})

stargazer.title("Regression Results Evacuation Distance ~ SDM + Didt_fire+ PDM ")
stargazer.custom_columns(["Model 1"], [1])
stargazer.show_degrees_of_freedom(False)                 
stargazer

In [ ]:
# latex_output = stargazer.render_latex()
# print(latex_output)

In [ ]:
# Extract model results
params = model_distance_OD_demo_dist_pdm.params.round(4)
pvalues = model_distance_OD_demo_dist_pdm.pvalues.round(4)
conf_int = model_distance_OD_demo_dist_pdm.conf_int().round(4)

# Define variable renaming (same as in Stargazer)
variable_names = {
    'Or_population_density': 'Population Density',
    'Or_unemployment_rate': 'Unemployment',
    'radius_of_gyration': 'Radius of Gyration',
    'log_Or_white_population': 'White Percent',
    'log_black_population': 'Black Percent',
    'log_Or_asian_population': 'Asian Percent',
    'avg_entropy': 'Entropy',
    'Or_median_household_income': 'Median Household Income',
    'Or_education_atleast_one_degree': 'Education Percent',
    'dist_epicentre': 'Distance from Fire'
}

# Start the LaTeX table
latex_output = r"""
\begin{table}[!htbp] \centering
  \caption{Regression Results: Evacuation Distance ~ SDM + Distance + PDM}
\begin{tabular}{@{\extracolsep{5pt}}lc}
\hline \hline
& \multicolumn{1}{c}{\textit{Dependent variable: Evacuation Distance}} \\
\cline{2-2}
\\[-1.8ex] & \textbf{Model 1} \\
\hline \\[-1.8ex]
"""

# Add coefficients, p-values, and confidence intervals
for var, coef in params.items():
    # Get the renamed variable, or default to the original name
    var_name = variable_names.get(var, var).title()
    pval = pvalues[var]
    lower_ci, upper_ci = conf_int.loc[var]
    
    # Add the row to the table
    latex_output += f"{var_name} & {coef} ($p={pval}$) \\\\\n"
    latex_output += f"& [{lower_ci}, {upper_ci}] \\\\\n"

# Add table footer
latex_output += r"""
\hline \\[-1.8ex]
Observations & %d \\
$R^2$ & %.3f \\
Adjusted $R^2$ & %.3f \\
\hline
\hline \\[-1.8ex]
\textit{Note:} & \multicolumn{1}{r}{$^{*}$p$<$0.1; $^{**}$p$<$0.05; $^{***}$p$<$0.01} \\
\end{tabular}
\end{table}
""" % (model_distance_OD_demo_dist_pdm.nobs, model_distance_OD_demo_dist_pdm.rsquared, model_distance_OD_demo_dist_pdm.rsquared_adj)

# Print the LaTeX output
print(latex_output)

### Regressions with plot

In [ ]:
df = evacuees_newSCI_cleaned.dropna()
df['log_scaled_sci'] = df['scaled_sci'].apply(np.log1p) 
Y_SCI = df['log_scaled_sci']
Y_homo = df['homophily']

df['log_radius_of_gyration'] = df['radius_of_gyration'].apply(np.log1p) 

# Independent variables for each model
models = {
    'Demographics (D1)': ['Or_population_density', 
                          'log_Or_unemployment_rate', 
                          'log_Or_white_population',
                          #'log_black_population',
                          #'log_Or_asian_population', 
                          #'log_Or_all_others',
                          'Or_median_household_income',
                          'Or_education_atleast_one_degree'
                         ],

    'D1 + Distance between OD (D2)': ['Or_population_density', 
                                      'log_Or_unemployment_rate',
                                      'log_Or_white_population',
                                      #'log_black_population',
                                      #'log_Or_asian_population',
                                      #'log_Or_all_others',
                                      'Or_median_household_income',
                                      'Or_education_atleast_one_degree', 
                                      'log_distance_OD'],

    'D1 + D2 + Distance from fire (D3)': ['Or_population_density', 
                                          'log_Or_unemployment_rate', 
                                          'log_Or_white_population',
                                          #'log_black_population',
                                          #'log_Or_asian_population',
                                          #'log_Or_all_others',
                                          'Or_median_household_income',
                                          'Or_education_atleast_one_degree', 
                                          'dist_epicentre',
                                          'log_distance_OD'],

    'D1 + D2 + D3 + Pre-disaster mobility': ['Or_population_density', 
                                             'log_Or_unemployment_rate', 
                                             'log_Or_white_population',
                                             #'log_black_population',
                                             #'log_Or_asian_population', 
                                             #'log_Or_all_others',
                                             'Or_median_household_income',
                                             'Or_education_atleast_one_degree', 
                                             'dist_epicentre',
                                             'log_distance_OD',
                                             'avg_entropy',
                                            'radius_of_gyration'
                                            ]
}

In [ ]:
Y_homo

In [ ]:
# Define the color palette
palette = ['#76FF7B', '#15B018', '#008000', '#054907']

# Initialize dictionaries to store R-squared values for both SCI and Homophily models
r_squared_values_sci = {}
r_squared_values_homo = {}

# Loop over the models to calculate R-squared values
for model_name, independent_vars in models.items():
    # Standardize the independent variables
    X = df[independent_vars].copy()
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    # Add a constant term
    X_with_const = sm.add_constant(X)

    # Calculate R-squared for SCI
    model_SCI = sm.OLS(Y_SCI, X_with_const).fit()
    r_squared_values_sci[model_name] = model_SCI.rsquared

    # Calculate R-squared for Homophily
    model_homophily = sm.OLS(Y_homo, X_with_const).fit()
    r_squared_values_homo[model_name] = model_homophily.rsquared

# Convert R-squared values to DataFrames for bar plots
r_squared_df_sci = pd.DataFrame(list(r_squared_values_sci.items()), columns=['Model', 'R-squared'])
r_squared_df_homo = pd.DataFrame(list(r_squared_values_homo.items()), columns=['Model', 'R-squared'])

# Plot horizontal R-squared bar plots for SCI using the custom palette
plt.figure(figsize=(5, 3))
sns.barplot(y='Model', x='R-squared', data=r_squared_df_sci, palette=palette)
plt.title("Familiarity")
plt.xlabel("R-squared")
plt.ylabel("Model")
plt.tight_layout()
plt.show()

# Plot horizontal R-squared bar plots for Homophily using the custom palette
plt.figure(figsize=(5, 3))
sns.barplot(y='Model', x='R-squared', data=r_squared_df_homo, palette=palette)
plt.title("Similarity")
plt.xlabel("R-squared")
plt.ylabel("Model")
plt.tight_layout()
plt.show()

In [ ]:
# Define the color palette
palette = ['#76FF7B', '#15B018', '#008000', '#054907']

# Define the order of variables for plotting
variable_labels = {
    'Or_population_density': 'Population\nDensity',
    'log_Or_unemployment_rate': 'Unemployment Rate',
    'log_Or_white_population': 'White\nPopulation',
    #'log_black_population': 'Black\nPopulation',
    #'log_Or_all_others' : 'All_other_races',
    #'log_Or_asian_population': 'Asian\nPopulation',
    'Or_median_household_income': 'Median\nIncome',
    'Or_education_atleast_one_degree': 'Education',
    'log_distance_OD': 'Distance origin\nto destination',
    'dist_epicentre': 'Distance\nfrom Fire',
    'avg_entropy': 'Entropy',
    'radius_of_gyration': 'Radius of\nGyration'
}

ordered_vars = list(variable_labels.keys())

# Assign colors to the variables based on their model groupings
color_mapping = {}

# Assign the first color to variables in the first model
for var in models['Demographics (D1)']:
    color_mapping[var] = palette[0]

# Assign the second color to variables added in the second model
for var in models['D1 + Distance between OD (D2)']:
    if var not in color_mapping:  # Avoid overwriting
        color_mapping[var] = palette[1]

# Assign the third color to variables added in the third model
for var in models['D1 + D2 + Distance from fire (D3)']:
    if var not in color_mapping:
        color_mapping[var] = palette[2]

# Assign the fourth color to variables added in the fourth model
for var in models['D1 + D2 + D3 + Pre-disaster mobility']:
    if var not in color_mapping:
        color_mapping[var] = palette[3]

def plot_significant_coefficients(df, Y, model_name, color_mapping, variable_labels, ordered_vars, significance_level=0.05):
    # Step 1: Get the independent variables for the selected model
    independent_vars = models[model_name]
    
    # Step 2: Standardize the independent variables
    X = df[independent_vars].copy()
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    # Step 3: Add a constant term for the intercept in the model
    X_with_const = sm.add_constant(X)

    # Step 4: Fit the OLS model using statsmodels
    model = sm.OLS(Y, X_with_const).fit()

    # Step 5: Create `coef_df` with coefficients, standard errors, and p-values
    coef_df = pd.DataFrame({
        'Variable': independent_vars,
        'Coefficient': model.params[1:],  # Exclude intercept
        'Standard_Error': model.bse[1:],  # Exclude intercept
        'P_value': model.pvalues[1:]  # Exclude intercept
    }).round(2)  # Round all values to 2 decimal points

    # Step 6: Filter for statistically significant coefficients only
    significant_coef_df = coef_df[coef_df['P_value'] < significance_level].copy()

    # Step 7: Ensure the variables in `significant_coef_df` are ordered as specified
    significant_coef_df['Variable'] = pd.Categorical(
        significant_coef_df['Variable'], categories=ordered_vars, ordered=True
    )
    significant_coef_df = significant_coef_df.sort_values('Variable')

    # Step 8: Plot significant coefficients with error bars
    fig, ax = plt.subplots(figsize=(4, 5))
    ax.invert_yaxis()

    # Plotting loop: Only plot variables that have mappings in `variable_labels` and `color_mapping`
    for i, row in significant_coef_df.iterrows():
        var_name = row['Variable']
        if var_name in variable_labels and var_name in color_mapping:
            ax.errorbar(row['Coefficient'], variable_labels[var_name], 
                        xerr=row['Standard_Error'], fmt='v', color=color_mapping[var_name],
                        ecolor='black', elinewidth=0.8, capsize=4, markersize=8, linestyle='None',
                        markerfacecolor=color_mapping[var_name], markeredgewidth=1.5, alpha=0.5)
        else:
            print(f"Warning: Missing mapping for {var_name}")

    # Add vertical line at zero and labels
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_xlabel('Beta Coefficients')
    ax.set_ylabel('Variables')
    ax.set_title(f'Significant Coefficients for {model_name}')
    plt.tight_layout()
    plt.show()

    # Print coefficients, standard errors, and p-values
    print("\nSignificant Coefficients Table:")
    print(significant_coef_df)

# Plot only significant coefficients for the "D1 + D2 + D3 + Pre-disaster mobility" model
plot_significant_coefficients(df, Y_SCI, 'D1 + D2 + D3 + Pre-disaster mobility', color_mapping, variable_labels, ordered_vars)
plot_significant_coefficients(df, Y_homo, 'D1 + D2 + D3 + Pre-disaster mobility', color_mapping, variable_labels, ordered_vars)


In [ ]:
# Calculate the additional contribution to R-squared for each model for log_D1_scaled_sci
contribution_sci = {}
prev_r2 = 0
for model_name, r2 in r_squared_values_sci.items():
    contribution_sci[model_name] = r2 - prev_r2
    prev_r2 = r2

# Calculate the additional contribution to R-squared for each model for log_homophily
contribution_homo = {}
prev_r2 = 0
for model_name, r2 in r_squared_values_homo.items():
    contribution_homo[model_name] = r2 - prev_r2
    prev_r2 = r2

# Convert the contributions to DataFrames for plotting
contribution_df_sci = pd.DataFrame(list(contribution_sci.items()), columns=['Model', 'Contribution'])
contribution_df_homo = pd.DataFrame(list(contribution_homo.items()), columns=['Model', 'Contribution'])

# Save the first plot (Familiarity) in 300 DPI
plt.figure(figsize=(5, 3))
sns.barplot(y='Model', x='Contribution', data=contribution_df_sci, palette=palette)
plt.title("Additional Contribution to R-squared (Familiarity)")
plt.xlabel("Additional R-squared")
plt.ylabel("Model")
plt.tight_layout()
plt.savefig('contribution_sci.png', dpi=300)
plt.show()

# Save the second plot (Similarity) in 300 DPI
plt.figure(figsize=(5, 3))
sns.barplot(y='Model', x='Contribution', data=contribution_df_homo, palette=palette)
plt.title("Additional Contribution to R-squared (Similarity)")
plt.xlabel("Additional R-squared")
plt.ylabel("Model")
plt.tight_layout()
plt.savefig('contribution_homo.png', dpi=300)
plt.show()

# Distance  plots

In [ ]:
from matplotlib.colors import Normalize

In [ ]:
# Define the columns to keep for correlation and plots
columns_keep = ['distance_OD', 'homophily', 'log_scaled_sci', 'log_distance_OD']

# Prepare the data for correlation analysis
data_for_corr = evacuees_newSCI_cleaned[columns_keep]
data_for_corr = data_for_corr[data_for_corr['log_distance_OD'] > 0]
data_for_corr.head()

In [ ]:
data_for_corr['homophily'].hist()

In [ ]:
# Define bins (0-10, 10-20, ..., 90-100)
bin_edges = np.linspace(0, 100000, 11)  # 11 edges for 10 bins
data_for_corr['distance_bin'] = pd.cut(
    data_for_corr['distance_OD'], 
    bins=bin_edges, 
    labels=[f'{int(bin_edges[i])}-{int(bin_edges[i+1])}' for i in range(len(bin_edges) - 1)]
)

# Create a numeric version of distance_bin for regression
data_for_corr['distance_bin_numeric'] = pd.Categorical(data_for_corr['distance_bin']).codes

# Function to plot boxplots with a straight trend line (linear regression)
def plot_boxplot_with_trend(data, variable, title, box_color, mean_line_color, trend_line_color, filename=None):
    plt.figure(figsize=(8, 6))
    
    # Create the boxplot with custom mean line color
    sns.boxplot(
        x='distance_bin', y=variable, data=data, showfliers=False,
        medianprops=dict(color=mean_line_color, linewidth=1.5), boxprops=dict(facecolor=box_color, alpha=1)
    )
    
    # Add the straight trend line using regplot
    sns.regplot(
        x='distance_bin_numeric', y=variable, data=data, scatter=False,
        color=trend_line_color, line_kws={'linewidth': 1.5}
    )
    
    plt.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.7)
    
    # Set plot labels
    plt.title(title, fontsize=14)
    plt.xlabel('Binned Distance', fontsize=12)
    plt.ylabel(variable, fontsize=12)

    # Save the plot if filename is provided
    if filename:
        plt.savefig(filename, dpi=300)

    plt.show()

# Define variables and colors
homophily_var = 'homophily'
sci_var = 'log_scaled_sci'

# Plot for homophily vs. binned distance
plot_boxplot_with_trend(
    data_for_corr, homophily_var, 
    f'Boxplot with Trend Line: {homophily_var} vs Binned Distance', 
    box_color='steelblue', mean_line_color='darkblue', trend_line_color='darkblue', filename='homophily_vs_distance.png'
)

# Plot for SCI vs. binned distance
plot_boxplot_with_trend(
    data_for_corr, sci_var, 
    f'Boxplot with Trend Line: {sci_var} vs Binned Distance', 
    box_color='orange', mean_line_color='chocolate', trend_line_color='chocolate', filename='sci_vs_distance.png'
)

In [ ]:
# Function to plot scatter plots with a trend line and additional annotations
def plot_scatter_with_trend(data, x_variable, y_variable, title, point_color, trend_line_color, y_limits=None, filename=None):
    plt.figure(figsize=(5, 3))  # Increase width for annotations
    
    # Scatter plot with transparent points
    sns.scatterplot(
        x=x_variable, y=y_variable, data=data, 
        color=point_color, alpha=0.1  # Transparency set to 10%
    )
    
    # Add the straight trend line using regplot and capture regression parameters
    reg_plot = sns.regplot(
        x=x_variable, y=y_variable, data=data, scatter=False,
        color=trend_line_color, line_kws={'linewidth': 1.5}
    )
    
    # Retrieve regression line coefficients
    line = reg_plot.get_lines()[0]  # Get the regression line
    x_vals = np.array(line.get_xdata())  # Extract x-values
    y_vals = np.array(line.get_ydata())  # Extract corresponding y-values
    
    # Calculate slope (beta) and intercept from the line's x and y data
    beta = round((y_vals[-1] - y_vals[0]) / (x_vals[-1] - x_vals[0]), 4)
    intercept = round(y_vals[0] - beta * x_vals[0], 4)
    
    # Calculate Pearson's correlation and p-value
    r, p_value = pearsonr(data[x_variable], data[y_variable])
    r = round(r, 4)
    p_value = round(p_value, 4)
    
    # Add annotations for r, p-value, beta, and intercept outside the plot
    plt.annotate(
        f"Pearson's r: {r}\nP-value: {p_value}\nBeta: {beta}\nIntercept: {intercept}",
        xy=(1.05, 0.5), xycoords='axes fraction', fontsize=10,
        horizontalalignment='left', verticalalignment='center',
        bbox=dict(boxstyle="round,pad=0.3", edgecolor='gray', facecolor='white')
    )
    
    # Apply y-axis limits if provided
    if y_limits:
        plt.ylim(y_limits)
    
    plt.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.7)
    
    # Set plot labels
    plt.title(title, fontsize=14)
    plt.xlabel(x_variable.replace('_', ' ').capitalize(), fontsize=12)
    plt.ylabel(y_variable.replace('_', ' ').capitalize(), fontsize=12)
    
    # Save the plot if filename is provided
    if filename:
        plt.savefig(filename, dpi=300, bbox_inches='tight')

    plt.show()

# Define variables and colors
distance_var = 'log_distance_OD'
homophily_var = 'homophily'
sci_var = 'log_scaled_sci'

# Plot for homophily vs. distance
plot_scatter_with_trend(
    data_for_corr, distance_var, homophily_var, 
    f'Similarity vs Distance', 
    point_color='steelblue', trend_line_color='steelblue', filename='homophily_vs_distance_scatter.png'
)

# Plot for SCI vs. distance with specific y-axis limits
plot_scatter_with_trend(
    data_for_corr, distance_var, sci_var, 
    f'Familiarity vs Distance', 
    point_color='orange', trend_line_color='orange', y_limits=(10, 18), filename='sci_vs_distance_scatter.png'
)